## Polars
#### Polars es una librería de análisis de datos en Python diseñada para manipular grandes volúmenes de datos de forma eficiente

In [1]:
import polars as pl
import datetime as dt

In [6]:
data = pl.DataFrame({
    'nombre': ['Ana Rodo', 'Carlos Ruiz', 'Maria Garcia', 'Jorge Martinez', 'Lucia Diaz',
               'Pedro Sanchez', 'Sofia Herrera', 'Diego Flores', 'Elena Castro', 'Miguel Vargas'],
    'cumpleaños': [
        dt.date(1990, 5, 15),
        dt.date(1985, 9, 20),
        dt.date(1992, 3, 10),
        dt.date(1988, 7, 25),
        dt.date(1995, 12, 8),
        dt.date(1987, 4, 18),
        dt.date(1993, 11, 3),
        dt.date(1986, 6, 12),
        dt.date(1991, 1, 22),
        dt.date(1989, 8, 17),
    ],
    'weight': [62.3, 78.9, 55.4, 85.0, 92.7, 65.3, 70.5, 60.8, 88.4, 76.9],
    'height': [1.68, 1.57, 1.74, 1.82, 1.90, 1.59, 1.77, 1.69, 1.85, 1.90],
    })

In [7]:
data.head()

nombre,cumpleaños,weight,height
str,date,f64,f64
"""Ana Rodo""",1990-05-15,62.3,1.68
"""Carlos Ruiz""",1985-09-20,78.9,1.57
"""Maria Garcia""",1992-03-10,55.4,1.74
"""Jorge Martinez""",1988-07-25,85.0,1.82
"""Lucia Diaz""",1995-12-08,92.7,1.9


In [8]:
data.tail()

nombre,cumpleaños,weight,height
str,date,f64,f64
"""Pedro Sanchez""",1987-04-18,65.3,1.59
"""Sofia Herrera""",1993-11-03,70.5,1.77
"""Diego Flores""",1986-06-12,60.8,1.69
"""Elena Castro""",1991-01-22,88.4,1.85
"""Miguel Vargas""",1989-08-17,76.9,1.9


#### Renombrando columnas

In [9]:
data = data.rename({'nombre': 'name', 'cumpleaños': 'birthday'})

In [11]:
data.columns

['name', 'birthday', 'weight', 'height']

## Metodos de Polars

### 1. Select

In [13]:
result = data.select(
    pl.col('name'),
    pl.col('birthday').dt.year().alias('year_birthday'),
   (pl.col('weight') / (pl.col('height')**2)).alias('BMI')

)
result

name,year_birthday,BMI
str,i32,f64
"""Ana Rodo""",1990,22.073413
"""Carlos Ruiz""",1985,32.009412
"""Maria Garcia""",1992,18.298322
"""Jorge Martinez""",1988,25.661152
"""Lucia Diaz""",1995,25.67867
"""Pedro Sanchez""",1987,25.829674
"""Sofia Herrera""",1993,22.503112
"""Diego Flores""",1986,21.28777
"""Elena Castro""",1991,25.829072


### 2. with_columns

In [14]:
result = data.with_columns(
    year_birthday = pl.col('birthday').dt.year(),
    BMI = (pl.col('weight') / (pl.col('height')**2))
)
result

name,birthday,weight,height,year_birthday,BMI
str,date,f64,f64,i32,f64
"""Ana Rodo""",1990-05-15,62.3,1.68,1990,22.073413
"""Carlos Ruiz""",1985-09-20,78.9,1.57,1985,32.009412
"""Maria Garcia""",1992-03-10,55.4,1.74,1992,18.298322
"""Jorge Martinez""",1988-07-25,85.0,1.82,1988,25.661152
"""Lucia Diaz""",1995-12-08,92.7,1.9,1995,25.67867
"""Pedro Sanchez""",1987-04-18,65.3,1.59,1987,25.829674
"""Sofia Herrera""",1993-11-03,70.5,1.77,1993,22.503112
"""Diego Flores""",1986-06-12,60.8,1.69,1986,21.28777
"""Elena Castro""",1991-01-22,88.4,1.85,1991,25.829072


### Seleccion de columnas e indice

In [15]:
result['name'].head(2)

name
str
"""Ana Rodo"""
"""Carlos Ruiz"""


In [16]:
result.select('name').head(2)

name
str
"""Ana Rodo"""
"""Carlos Ruiz"""


In [17]:
result[:3]

name,birthday,weight,height,year_birthday,BMI
str,date,f64,f64,i32,f64
"""Ana Rodo""",1990-05-15,62.3,1.68,1990,22.073413
"""Carlos Ruiz""",1985-09-20,78.9,1.57,1985,32.009412
"""Maria Garcia""",1992-03-10,55.4,1.74,1992,18.298322


In [26]:
result = data.filter(
    pl.col('birthday').dt.year() > 1993
)
result

name,birthday,weight,height
str,date,f64,f64
"""Lucia Diaz""",1995-12-08,92.7,1.9


In [32]:
result = data.filter(
    pl.col('birthday').is_between(dt.date(1993,1,1), dt.date(2000,12,31)),
    pl.col('weight') > 80
)
result


name,birthday,weight,height
str,date,f64,f64
"""Lucia Diaz""",1995-12-08,92.7,1.9


### group_by

In [35]:
result = data.group_by(
    (pl.col('birthday').dt.year() // 10 * 10).alias('decade'),
    maintain_order = True
).len()
result

decade,len
i32,u32
1990,5
1980,5


#### Ordenamos por cumpleaños en decadas y ñuegp agregamos peso promedio y altura promedio

In [38]:
result = data.group_by(
    (pl.col('birthday').dt.year() // 10 * 10).alias('decade'),
    maintain_order = True
).agg(
    pl.len().alias('Tamaño_muestra'),
    pl.col('weight').mean().round(2).alias('Peso_promedio'),
    pl.col('height').mean().round(2).alias('Altura_promedio'),
)
result

decade,Tamaño_muestra,Peso_promedio,Altura_promedio
i32,u32,f64,f64
1990,5,73.86,1.79
1980,5,73.38,1.71


### Combinar dataFrames

In [42]:
data2 = pl.DataFrame({
    'name': ['Ana Rodo', 'Carlos Ruiz', 'Maria Garcia', 'Jorge Martinez', 'Lucia Diaz'],
    'birthday': [
        dt.date(1995, 7, 12),
        dt.date(1990, 3, 28),
        dt.date(1988, 11, 25),
        dt.date(1992, 5, 18),
        dt.date(1998, 9, 10),
    ],
    'weight': [62.3, 78.5, 55.4, 85.0, 59.1],
    'height': [1.68, 1.57, 1.74, 1.82, 1.65],
})
data2.head()

name,birthday,weight,height
str,date,f64,f64
"""Ana Rodo""",1995-07-12,62.3,1.68
"""Carlos Ruiz""",1990-03-28,78.5,1.57
"""Maria Garcia""",1988-11-25,55.4,1.74
"""Jorge Martinez""",1992-05-18,85.0,1.82
"""Lucia Diaz""",1998-09-10,59.1,1.65


In [45]:
pl.concat([data, data2], how='vertical')

name,birthday,weight,height
str,date,f64,f64
"""Ana Rodo""",1990-05-15,62.3,1.68
"""Carlos Ruiz""",1985-09-20,78.9,1.57
"""Maria Garcia""",1992-03-10,55.4,1.74
"""Jorge Martinez""",1988-07-25,85.0,1.82
"""Lucia Diaz""",1995-12-08,92.7,1.9
…,…,…,…
"""Ana Rodo""",1995-07-12,62.3,1.68
"""Carlos Ruiz""",1990-03-28,78.5,1.57
"""Maria Garcia""",1988-11-25,55.4,1.74


In [47]:
data3 = pl.DataFrame({
    'name': ['Ana Rodo', 'Carlos Ruiz', 'Maria Garcia', 'Jorge Martinez', 'Lucia Diaz'],
    'ciudad':['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Malaga'],
    'salario':[45000, 656820, 78000, 898655, 786454],
    'departamento': ['Marketing', 'Ventas', 'Finanzas', 'Ventas', 'Marketing'],
})
data3

name,ciudad,salario,departamento
str,str,i64,str
"""Ana Rodo""","""Madrid""",45000,"""Marketing"""
"""Carlos Ruiz""","""Barcelona""",656820,"""Ventas"""
"""Maria Garcia""","""Valencia""",78000,"""Finanzas"""
"""Jorge Martinez""","""Sevilla""",898655,"""Ventas"""
"""Lucia Diaz""","""Malaga""",786454,"""Marketing"""


In [48]:
data2.join(data3, on='name')

name,birthday,weight,height,ciudad,salario,departamento
str,date,f64,f64,str,i64,str
"""Ana Rodo""",1995-07-12,62.3,1.68,"""Madrid""",45000,"""Marketing"""
"""Carlos Ruiz""",1990-03-28,78.5,1.57,"""Barcelona""",656820,"""Ventas"""
"""Maria Garcia""",1988-11-25,55.4,1.74,"""Valencia""",78000,"""Finanzas"""
"""Jorge Martinez""",1992-05-18,85.0,1.82,"""Sevilla""",898655,"""Ventas"""
"""Lucia Diaz""",1998-09-10,59.1,1.65,"""Malaga""",786454,"""Marketing"""


In [49]:
data2.join(data3, on='name', how='inner')

name,birthday,weight,height,ciudad,salario,departamento
str,date,f64,f64,str,i64,str
"""Ana Rodo""",1995-07-12,62.3,1.68,"""Madrid""",45000,"""Marketing"""
"""Carlos Ruiz""",1990-03-28,78.5,1.57,"""Barcelona""",656820,"""Ventas"""
"""Maria Garcia""",1988-11-25,55.4,1.74,"""Valencia""",78000,"""Finanzas"""
"""Jorge Martinez""",1992-05-18,85.0,1.82,"""Sevilla""",898655,"""Ventas"""
"""Lucia Diaz""",1998-09-10,59.1,1.65,"""Malaga""",786454,"""Marketing"""


In [50]:
data.join(data3, on='name', how='left')

name,birthday,weight,height,ciudad,salario,departamento
str,date,f64,f64,str,i64,str
"""Ana Rodo""",1990-05-15,62.3,1.68,"""Madrid""",45000,"""Marketing"""
"""Carlos Ruiz""",1985-09-20,78.9,1.57,"""Barcelona""",656820,"""Ventas"""
"""Maria Garcia""",1992-03-10,55.4,1.74,"""Valencia""",78000,"""Finanzas"""
"""Jorge Martinez""",1988-07-25,85.0,1.82,"""Sevilla""",898655,"""Ventas"""
"""Lucia Diaz""",1995-12-08,92.7,1.9,"""Malaga""",786454,"""Marketing"""
"""Pedro Sanchez""",1987-04-18,65.3,1.59,null,null,null
"""Sofia Herrera""",1993-11-03,70.5,1.77,null,null,null
"""Diego Flores""",1986-06-12,60.8,1.69,null,null,null
"""Elena Castro""",1991-01-22,88.4,1.85,null,null,null


In [51]:
data.join(data3, on='name', how='right')

birthday,weight,height,name,ciudad,salario,departamento
date,f64,f64,str,str,i64,str
1990-05-15,62.3,1.68,"""Ana Rodo""","""Madrid""",45000,"""Marketing"""
1985-09-20,78.9,1.57,"""Carlos Ruiz""","""Barcelona""",656820,"""Ventas"""
1992-03-10,55.4,1.74,"""Maria Garcia""","""Valencia""",78000,"""Finanzas"""
1988-07-25,85.0,1.82,"""Jorge Martinez""","""Sevilla""",898655,"""Ventas"""
1995-12-08,92.7,1.9,"""Lucia Diaz""","""Malaga""",786454,"""Marketing"""


### Pureba en pandas y polars

In [52]:
import pandas as pd
import time

In [59]:
inicio = time.time()
data = pd.read_csv('/content/ModalidadVirtual.csv')
print(time.time() - inicio)

0.0045108795166015625


In [60]:
inicio = time.time()
data = pl.read_csv('/content/ModalidadVirtual.csv')
print(time.time() - inicio)

0.0019445419311523438
